> **🧭 Navigasi Proyek CRISP-DM:**
> [🏠 00. Business Understanding](./00_Index_dan_Business_Understanding.ipynb) | [📊 01. Data Understanding & EDA](./01_Data_Understanding_dan_EDA.ipynb) | [⚙️ 02. Data Preparation & 32 Fitur](./02_Data_Preparation_Chunking_dan_Ekstraksi_32Fitur.ipynb) | [🧠 03. Modeling & Benchmarking](./03_Modeling_dan_Benchmarking.ipynb) | [🚀 04. Evaluation & Deployment](./04_Evaluation_dan_Simulasi_Deployment.ipynb)

# 🎙️ CrowdSense Audio: Sistem Deteksi Tingkat Keramaian Ruangan Berbasis Sinyal Akustik
## FASE 1 CRISP-DM: Index Proyek & Business Understanding

---

## 📑 Master Table of Contents (Daftar Isi Proyek)
Rangkaian notebook ini mendokumentasikan implementasi *end-to-end* sains data berdasarkan metodologi standar industri **CRISP-DM (*Cross-Industry Standard Process for Data Mining*)**:

1. [🏠 **Notebook 00: Index & Business Understanding**](./00_Index_dan_Business_Understanding.ipynb) (Halaman Ini)
   - Latar belakang masalah, motivasi solusi akustik vs CCTV, sasaran bisnis vs teknis, dan arsitektur sistem.
2. [📊 **Notebook 01: Data Understanding & EDA**](./01_Data_Understanding_dan_EDA.ipynb)
   - Eksplorasi karakteristik audio mentah, waveform domain waktu, mel-spectrogram domain frekuensi, dan sebaran durasi.
3. [⚙️ **Notebook 02: Data Preparation, Chunking & Ekstraksi 32 Fitur**](./02_Data_Preparation_Chunking_dan_Ekstraksi_32Fitur.ipynb)
   - Standarisasi 16 kHz mono, safe silence trimming, pemotongan 5 detik anti-leakage, ekstraksi 32 fitur DSP Librosa, boxplot, dan korelasi.
4. [🧠 **Notebook 03: Modeling & Benchmarking**](./03_Modeling_dan_Benchmarking.ipynb)
   - Pemisahan data anti-kebocoran (`GroupShuffleSplit`), 5-Fold GroupKFold Cross-Validation, pelatihan dan benchmarking 4 algoritma (Random Forest, SVM, Gradient Boosting, Logistic Regression).
5. [🚀 **Notebook 04: Evaluation & Simulasi Deployment**](./04_Evaluation_dan_Simulasi_Deployment.ipynb)
   - Confusion Matrix, kurva ROC-AUC, Top 10 Feature Importance, simulasi inferensi live per sampel, dan panduan deployment Streamlit (`app.py`).

### 🔗 Tautan Eksternal & Artefak Terkait:
- 📱 [Aplikasi Web Interaktif Streamlit (`app.py`)](../app.py)
- 📖 [Dokumentasi Lengkap Proyek (`README.md`)](../README.md)
- 📑 [Laporan Akademis CRISP-DM & Panduan 10 Slide Sidang (`LAPORAN_CRISPDM.md`)](../LAPORAN_CRISPDM.md)
- ⚙️ [Spesifikasi Pustaka Dependensi (`requirements.txt`)](../requirements.txt)


---
## 🏛️ Fase 1 CRISP-DM: Business Understanding

### 1.1. Latar Belakang & Perumusan Masalah
Pengelolaan fasilitas umum kampus seperti perpustakaan, *co-working space*, ruang diskusi, dan laboratorium memerlukan pemantauan tingkat kebisingan secara berkala demi menjamin kenyamanan belajar. Dua pendekatan yang ada saat ini memiliki kelemahan fatal:
1. **Patroli Manual Petugas:** Sangat memakan waktu, subjektif, dan tidak mungkin mengawasi puluhan ruangan secara terus-menerus 24 jam.
2. **Kamera CCTV + Computer Vision:**
   - **Pelanggaran Privasi Pengunjung:** Pengunjung merasa risih dan diawasi gerak-geriknya secara visual (*privacy concern*).
   - **Beban Biaya & Komputasi Tinggi:** Membutuhkan kamera resolusi tinggi, bandwidth jaringan besar, penyimpanan terabita, serta GPU mahal untuk memproses model deteksi orang (*YOLO/CNN*).

### 1.2. Solusi yang Diusulkan: Akustik Non-Invasif (*Privacy-Preserving*)
Kami membangun sistem klasifikasi kebisingan ruangan berbasis **sensor audio mikrofon** yang secara instan mereduksi sinyal suara menjadi **32 parameter statistik numerik** (RMS, Spectral Centroid, ZCR, dan MFCC).
- **100% Ramah Privasi (*Privacy-Preserving*):** Sinyal audio vokal manusia tidak pernah disimpan, direkam, maupun ditranskripsikan.
- **Sangat Ringan:** Berjalan mulus di CPU laptop/server mini biasa dengan waktu inferensi belasan milidetik.

### 1.3. Matriks Kriteria Keberhasilan Terukur

| Aspek | Parameter | Target Terukur (*Baseline Goal*) |
| :--- | :--- | :--- |
| **Kriteria Bisnis** | Efisiensi Kerja | Menggantikan patroli manual dengan alarm deteksi otomatis. |
| | Perlindungan Privasi | Bebas penyimpanan audio percakapan pribadi. |
| **Kriteria Sains Data** | Akurasi Klasifikasi | $\text{Accuracy} \ge 80.0\%$ pada data uji (*unseen test data*). |
| | Metrik F1-Score | $\text{F1-Score} \ge 0.85$ pada validasi silang (*5-Fold CV*). |
| | Kecepatan Inferensi | $\text{Latency} < 1.000\text{ ms}$ ($< 1\text{ detik}$) per sampel audio. |
| | Integritas Metodologis | Zero Data Leakage (pembagian data berbasis file rekaman asli). |

### 1.4. Diagram Arsitektur Alur Sistem
```text
[Sinyal Suara Ruangan]
         │
         ▼
[Preprocessing Audio] ──> Resampling 16kHz Mono + Safe Trimming
         │
         ▼
[Jendela 5 Detik] ──────> 80.000 Sampel (Zero-padding jika sisa >= 2.5s)
         │
         ▼
[Ekstraksi 32 Fitur] ───> RMS (2) + Centroid (2) + ZCR (2) + MFCC 1-13 (26)
         │
         ▼
[Random Forest Model] ──> Majority Voting 100 Decision Trees
         │
         ▼
[Keputusan Deteksi] ───> 🔴 RAMAI (Kelas 1) vs 🟢 TIDAK RAMAI (Kelas 0)
```


In [1]:
import os
import sys
import glob
import pandas as pd
import numpy as np
import librosa
import soundfile
import sklearn

print("=" * 65)
print(" VERIFIKASI ENVIRONMENT & DATASET MENTAH")
print("=" * 65)
print(f"Python Executable : {sys.executable}")
print(f"Python Version    : {sys.version.split()[0]}")
print(f"Librosa Version   : {librosa.__version__}")
print(f"SoundFile Version : {soundfile.__version__}")
print(f"Scikit-Learn      : {sklearn.__version__}")
print(f"Pandas Version    : {pd.__version__}")
print(f"NumPy Version     : {np.__version__}")

# Verifikasi folder data mentah
raw_ramai_dir = os.path.join("..", "data", "01_raw", "ramai")
raw_sepi_dir = os.path.join("..", "data", "01_raw", "tidak_ramai")
mapping_path = os.path.join("..", "data", "01_raw", "file_mapping.csv")

ramai_files = glob.glob(os.path.join(raw_ramai_dir, "*.*"))
sepi_files = glob.glob(os.path.join(raw_sepi_dir, "*.*"))

print("\n--- RINGKASAN FILE REKAMAN ASLI (01_RAW) ---")
print(f"Jumlah file di kelas 'Ramai'       : {len(ramai_files)} file")
print(f"Jumlah file di kelas 'Tidak Ramai' : {len(sepi_files)} file")
print(f"Total file mentah terdata          : {len(ramai_files) + len(sepi_files)} file")

if os.path.exists(mapping_path):
    map_df = pd.read_csv(mapping_path)
    print(f"Tabel pemetaan file_mapping.csv    : {map_df.shape[0]} baris tercatat.")
    display(map_df.head(6))
else:
    print("file_mapping.csv belum ditemukan.")


 VERIFIKASI ENVIRONMENT & DATASET MENTAH
Python Executable : C:\Users\heker\AppData\Local\Programs\Python\Python313\python.exe
Python Version    : 3.13.5
Librosa Version   : 1.0.0
SoundFile Version : 0.14.0
Scikit-Learn      : 1.9.0
Pandas Version    : 2.3.3
NumPy Version     : 2.2.6

--- RINGKASAN FILE REKAMAN ASLI (01_RAW) ---
Jumlah file di kelas 'Ramai'       : 101 file
Jumlah file di kelas 'Tidak Ramai' : 94 file
Total file mentah terdata          : 195 file
Tabel pemetaan file_mapping.csv    : 195 baris tercatat.


,class,parent_id,original_filename,standardized_filename,original_path,raw_path
0,ramai,ramai_001,074_berisik.mpeg,ramai_001.mpeg,AUDIO/drive-download-20261019T005432Z-1-001\07...,data/01_raw/ramai\ramai_001.mpeg
1,ramai,ramai_002,074_berisikk.mpeg,ramai_002.mpeg,AUDIO/drive-download-20261019T005432Z-1-001\07...,data/01_raw/ramai\ramai_002.mpeg
2,ramai,ramai_003,074_berisikkk.mpeg,ramai_003.mpeg,AUDIO/drive-download-20261019T005432Z-1-001\07...,data/01_raw/ramai\ramai_003.mpeg
3,ramai,ramai_004,24-003 berisik (1).mp3,ramai_004.mp3,AUDIO/drive-download-20261019T005432Z-1-001\24...,data/01_raw/ramai\ramai_004.mp3
4,ramai,ramai_005,24-003 berisik (2).mp3,ramai_005.mp3,AUDIO/drive-download-20261019T005432Z-1-001\24...,data/01_raw/ramai\ramai_005.mp3
5,ramai,ramai_006,24-003 berisik (3).mp3,ramai_006.mp3,AUDIO/drive-download-20261019T005432Z-1-001\24...,data/01_raw/ramai\ramai_006.mp3


> **🧭 Navigasi Proyek CRISP-DM:**
> [🏠 00. Business Understanding](./00_Index_dan_Business_Understanding.ipynb) | [📊 01. Data Understanding & EDA](./01_Data_Understanding_dan_EDA.ipynb) | [⚙️ 02. Data Preparation & 32 Fitur](./02_Data_Preparation_Chunking_dan_Ekstraksi_32Fitur.ipynb) | [🧠 03. Modeling & Benchmarking](./03_Modeling_dan_Benchmarking.ipynb) | [🚀 04. Evaluation & Deployment](./04_Evaluation_dan_Simulasi_Deployment.ipynb)